<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보충 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 6장: 텍스트 분류를 위한 미세조정(Finetuning for Text Classification)

In [ ]:
from importlib.metadata import version

pkgs = ["matplotlib",  # 플롯팅 라이브러리
        "numpy",       # PyTorch & TensorFlow 종속성
        "tiktoken",    # 토크나이저
        "torch",       # 딥러닝 라이브러리
        "tensorflow",  # OpenAI의 사전훈련된 가중치를 위해
        "pandas"       # 데이터셋 로딩
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/chapter-overview.webp" width=500px>

## 6.1 미세조정의 다양한 범주(Different categories of finetuning)

- 이 섹션에는 코드가 없습니다

- 언어 모델을 미세조정하는 가장 일반적인 방법은 지시 미세조정(instruction-finetuning)과 분류 미세조정(classification finetuning)입니다
- 아래에 보여진 지시 미세조정은 다음 장의 주제입니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/instructions.webp" width=500px>

- 이 장의 주제인 분류 미세조정은 머신러닝 배경지식이 있다면 이미 익숙할 수 있는 절차입니다 -- 예를 들어, 손글씨 숫자를 분류하기 위해 컨볼루션 네트워크를 훈련하는 것과 유사합니다
- 분류 미세조정에서는 모델이 출력할 수 있는 특정한 수의 클래스 레이블(예: "스팸"과 "스팸 아님")이 있습니다
- 분류 미세조정된 모델은 훈련 중에 본 클래스들(예: "스팸" 또는 "스팸 아님")만 예측할 수 있는 반면, 지시 미세조정된 모델은 일반적으로 많은 작업을 수행할 수 있습니다
- 분류 미세조정된 모델을 매우 특화된 모델로 생각할 수 있습니다; 실제로는 여러 다른 작업을 잘 수행하는 일반화된 모델보다 특화된 모델을 만드는 것이 훨씬 쉽습니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/spam-non-spam.webp" width=500px>

## 6.2 데이터셋 준비(Preparing the dataset)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/overview-1.webp" width=500px>

- 이 섹션에서는 분류 미세조정에 사용할 데이터셋을 준비합니다
- 스팸과 비스팸 텍스트 메시지로 구성된 데이터셋을 사용하여 LLM을 분류하도록 미세조정합니다
- 먼저 데이터셋을 다운로드하고 압축을 해제합니다

In [ ]:
import urllib.request
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} 파일이 이미 존재합니다. 다운로드 및 압축 해제를 건너뜁니다.")
        return

    # 파일 다운로드
    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    # 파일 압축 해제
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    # .tsv 파일 확장자 추가
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"파일이 다운로드되어 {data_file_path}로 저장되었습니다")

try:
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError) as e:
    print(f"주 URL이 실패했습니다: {e}. 백업 URL을 시도합니다...")
    url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/sms%2Bspam%2Bcollection.zip"
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

- 데이터셋은 탭으로 분리된 텍스트 파일로 저장되며, 이를 pandas DataFrame으로 로드할 수 있습니다

In [ ]:
import pandas as pd

df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
df

- 클래스 분포를 확인해보면, 데이터에는 "ham"(즉, "스팸 아님")이 "spam"보다 훨씬 더 빈번하게 포함되어 있음을 볼 수 있습니다

In [ ]:
print(df["Label"].value_counts())

- 단순화를 위해, 그리고 어차피 교육 목적으로 작은 데이터셋을 선호하기 때문에 (LLM을 더 빠르게 미세조정할 수 있게 해줍니다), 각 클래스에서 747개의 인스턴스를 포함하도록 데이터셋을 부분표집(언더샘플링)합니다
- (언더샘플링 외에도 클래스 불균형을 다루는 여러 다른 방법들이 있지만, 이들은 LLM에 관한 책의 범위를 벗어납니다; [`imbalanced-learn` 사용자 가이드](https://imbalanced-learn.org/stable/user_guide.html)에서 예제와 더 많은 정보를 찾을 수 있습니다)

In [ ]:
def create_balanced_dataset(df):
    
    # "spam" 인스턴스 수 계산
    num_spam = df[df["Label"] == "spam"].shape[0]
    
    # "spam" 인스턴스 수와 맞추기 위해 "ham" 인스턴스를 무작위로 샘플링
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)
    
    # ham "부분집합"과 "spam"을 결합
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])

    return balanced_df


balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

- 다음으로, 문자열 클래스 레이블인 "ham"과 "spam"을 정수 클래스 레이블인 0과 1로 변경합니다:

In [ ]:
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

In [ ]:
balanced_df

- 이제 데이터셋을 훈련, 검증, 테스트 부분집합으로 무작위로 나누는 함수를 정의하겠습니다

In [ ]:
def random_split(df, train_frac, validation_frac):
    # 전체 DataFrame을 셔플
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    # 분할 인덱스 계산
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    # DataFrame 분할
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)
# 테스트 크기는 나머지로 0.2로 암시됨

train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

## 6.3 데이터 로더 생성(Creating data loaders)

- 텍스트 메시지들의 길이가 다르다는 점에 주목하세요; 여러 훈련 예제를 배치로 결합하려면, 다음 중 하나를 해야 합니다:
  1. 모든 메시지를 데이터셋 또는 배치에서 가장 짧은 메시지의 길이로 잘라내거나
  2. 모든 메시지를 데이터셋 또는 배치에서 가장 긴 메시지의 길이로 패딩하거나

- 2번 옵션을 선택하고 모든 메시지를 데이터셋에서 가장 긴 메시지로 패딩합니다
- 이를 위해 2장에서 논의한 대로 `<|endoftext|>`를 패딩 토큰으로 사용합니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/pad-input-sequences.webp?123" width=500px>

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

- 아래의 `SpamDataset` 클래스는 훈련 데이터셋에서 가장 긴 시퀀스를 식별하고 다른 시퀀스들에 패딩 토큰을 추가하여 그 시퀀스 길이와 맞춥니다

In [ ]:
import torch
from torch.utils.data import Dataset


class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        # 텍스트를 사전 토큰화
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            # max_length보다 긴 시퀀스는 잘라냄
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        # 가장 긴 시퀀스로 시퀀스들을 패딩
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length
        # 참고: 이 메소드를 구현하는 더 파이썬다운 버전은
        # 다음과 같으며, 다음 장에서도 사용됩니다:
        # return max(len(encoded_text) for encoded_text in self.encoded_texts)

In [ ]:
train_dataset = SpamDataset(
    csv_file="train.csv",
    max_length=None,
    tokenizer=tokenizer
)

print(train_dataset.max_length)

- 검증 및 테스트 세트도 가장 긴 훈련 시퀀스로 패딩합니다
- 가장 긴 훈련 예제보다 긴 검증 및 테스트 세트 샘플들은 `SpamDataset` 코드의 `encoded_text[:self.max_length]`를 통해 잘립니다
- 이 동작은 전적으로 선택적이며, 검증 및 테스트 세트 경우 모두에서 `max_length=None`으로 설정해도 잘 작동합니다

In [ ]:
val_dataset = SpamDataset(
    csv_file="validation.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)
test_dataset = SpamDataset(
    csv_file="test.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

- 다음으로, 데이터셋을 사용하여 데이터 로더를 인스턴스화하는데, 이는 이전 장들에서 데이터 로더를 생성하는 것과 유사합니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/batch.webp" width=500px>

In [ ]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

- 검증 단계로서, 데이터 로더들을 반복하여 배치가 각각 8개의 훈련 예제를 포함하고, 각 훈련 예제가 120개의 토큰으로 구성되어 있음을 확인합니다

In [ ]:
print("훈련 로더:")
for input_batch, target_batch in train_loader:
    pass

print("입력 배치 차원:", input_batch.shape)
print("라벨 배치 차원:", target_batch.shape)

- 마지막으로, 각 데이터셋의 총 배치 수를 출력해보겠습니다

In [ ]:
print(f"{len(train_loader)} 훈련 배치")
print(f"{len(val_loader)} 검증 배치")
print(f"{len(test_loader)} 테스트 배치")

## 6.4 사전훈련된 가중치로 모델 초기화(Initializing a model with pretrained weights)

- 이 섹션에서는 이전 장에서 작업했던 사전훈련된 모델을 초기화합니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/overview-2.webp" width=500px>

In [ ]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257,     # 어휘 크기
    "context_length": 1024,  # 컨텍스트 길이
    "drop_rate": 0.0,        # 드롭아웃 비율
    "qkv_bias": True         # 쿼리-키-값 편향
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

assert train_dataset.max_length <= BASE_CONFIG["context_length"], (
    f"데이터셋 길이 {train_dataset.max_length}가 모델의 컨텍스트 "
    f"길이 {BASE_CONFIG['context_length']}를 초과합니다. "
    f"`max_length={BASE_CONFIG['context_length']}`로 데이터 세트를 재초기화하세요"
)

In [ ]:
from gpt_download import download_and_load_gpt2
from previous_chapters import GPTModel, load_weights_into_gpt
# `previous_chapters.py` 파일이 로컬에서 사용할 수 없는 경우,
# `llms-from-scratch` PyPI 패키지에서 가져올 수 있습니다.
# 자세한 내용은: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# 예:
# from llms_from_scratch.ch04 import GPTModel
# from llms_from_scratch.ch05 import download_and_load_gpt2, load_weights_into_gpt

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval();

- 모델이 올바르게 로드되었는지 확인하기 위해, 일관된 텍스트를 생성하는지 다시 한 번 확인해보겠습니다

In [ ]:
from previous_chapters import (
    generate_text_simple,
    text_to_token_ids,
    token_ids_to_text
)

# 대안적으로:
# from llms_from_scratch.ch05 import (
#    generate_text_simple,
#    text_to_token_ids,
#    token_ids_to_text
# )


text_1 = "Every effort moves you"

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

- 모델을 분류기로 미세조정하기 전에, 모델이 프롬프팅을 통해 스팸 메시지를 분류할 수 있는지 확인해보겠습니다

In [ ]:
text_2 = (
    "다음 텍스트가 '스팸'인가요? 'yes' 또는 'no'로 답하세요:"
    " '당신은 당첨자입니다. 특별히 선택되어"
    " $1000 현금 또는 $2000 상품을 받으실 수 있습니다.'"
)

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_2, tokenizer),
    max_new_tokens=23,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

- 보시다시피, 모델은 지시를 따르는 데 별로 좋지 않습니다
- 이는 예상된 결과입니다. 모델이 사전훈련만 되어 있고 지시 미세조정이 되어 있지 않기 때문입니다 (지시 미세조정은 다음 장에서 다룰 예정입니다)

## 6.5 분류 헤드 추가(Adding a classification head)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/lm-head.webp" width=500px>

- 이 섹션에서는 분류 미세조정을 위해 준비하도록 사전훈련된 LLM을 수정합니다
- 먼저 모델 아키텍처를 살펴보겠습니다

In [ ]:
print(model)

- 위에서 4장에서 구현한 아키텍처가 깔끔하게 배치된 것을 볼 수 있습니다
- 목표는 출력 레이어를 교체하고 미세조정하는 것입니다
- 이를 달성하기 위해, 먼저 모델을 동결합니다. 즉, 모든 레이어를 훈련 불가능하게 만듭니다

In [ ]:
for param in model.parameters():
    param.requires_grad = False

- 그런 다음, 원래 레이어 입력을 50,257 차원(어휘의 크기)으로 매핑하는 출력 레이어(`model.out_head`)를 교체합니다
- 이진 분류("스팸"과 "스팸 아님" 2개 클래스 예측)를 위해 모델을 미세조정하므로, 아래 표시된 대로 출력 레이어를 교체할 수 있으며, 이는 기본적으로 훈련 가능합니다
- 코드를 더 일반적으로 만들기 위해 `BASE_CONFIG["emb_dim"]`("gpt2-small (124M)" 모델에서 768과 같음)을 사용한다는 점에 주목하세요

In [ ]:
torch.manual_seed(123)

num_classes = 2
model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=num_classes)

- 기술적으로는 출력 레이어만 훈련하는 것으로 충분합니다
- 하지만 [Finetuning Large Language Models](https://magazine.sebastianraschka.com/p/finetuning-large-language-models)에서 발견한 바와 같이, 실험 결과 추가 레이어를 미세조정하면 성능이 눈에 띄게 개선될 수 있습니다
- 따라서 마지막 트랜스포머 블록과 마지막 트랜스포머 블록을 출력 레이어에 연결하는 최종 `LayerNorm` 모듈도 훈련 가능하게 만듭니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/trainable.webp" width=500px>

In [ ]:
for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True

for param in model.final_norm.parameters():
    param.requires_grad = True

- 여전히 이전 장들에서와 유사하게 이 모델을 사용할 수 있습니다
- 예를 들어, 텍스트 입력을 제공해보겠습니다

In [ ]:
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print("입력:", inputs)
print("입력 차원:", inputs.shape) # 형태: (batch_size, num_tokens)

- 이전 장들과 다른 점은 이제 50,257개가 아닌 2개의 출력 차원을 가진다는 것입니다

In [ ]:
with torch.no_grad():
    outputs = model(inputs)

print("출력:\n", outputs)
print("출력 차원:", outputs.shape) # 형태: (batch_size, num_tokens, num_classes)

- 이전 장들에서 논의한 바와 같이, 각 입력 토큰에 대해 하나의 출력 벡터가 있습니다
- 4개의 입력 토큰을 가진 텍스트 샘플을 모델에 제공했으므로, 출력은 위에서와 같이 4개의 2차원 출력 벡터로 구성됩니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/input-and-output.webp" width=500px>

- 3장에서는 각 입력 토큰을 다른 모든 입력 토큰에 연결하는 어텐션 메커니즘에 대해 논의했습니다
- 3장에서는 GPT와 같은 모델에서 사용되는 인과적 어텐션 마스크도 소개했습니다; 이 인과적 마스크는 현재 토큰이 현재 및 이전 토큰 위치에만 주의를 기울이게 합니다
- 이 인과적 어텐션 메커니즘을 기반으로, 4번째(마지막) 토큰은 다른 모든 토큰에 대한 정보를 포함하는 유일한 토큰이기 때문에 모든 토큰 중에서 가장 많은 정보를 담고 있습니다
- 따라서 우리는 스팸 분류 작업을 위해 미세조정할 이 마지막 토큰에 특별히 관심이 있습니다

In [ ]:
print("마지막 출력 토큰:", outputs[:, -1, :])

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/attention-mask.webp" width=200px>

## 6.6 분류 손실과 정확도 계산(Calculating the classification loss and accuracy)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/overview-3.webp?1" width=500px>

- 손실 계산을 설명하기 전에, 모델 출력이 클래스 라벨로 어떻게 변환되는지 간략히 살펴보겠습니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/class-argmax.webp" width=600px>

In [ ]:
print("마지막 출력 토큰:", outputs[:, -1, :])

- 5장과 유사하게, `softmax` 함수를 통해 출력(로짓)을 확률 점수로 변환한 다음, `argmax` 함수를 통해 가장 큰 확률 값의 인덱스 위치를 구합니다

In [ ]:
probas = torch.softmax(outputs[:, -1, :], dim=-1)
label = torch.argmax(probas)
print("클래스 라벨:", label.item())

- 5장에서 설명한 바와 같이, 가장 큰 출력이 가장 큰 확률 점수에 해당하기 때문에 여기서 softmax 함수는 선택사항이라는 점에 주목하세요

In [ ]:
logits = outputs[:, -1, :]
label = torch.argmax(logits)
print("클래스 라벨:", label.item())

- 이 개념을 적용하여 소위 분류 정확도를 계산할 수 있습니다. 분류 정확도는 주어진 데이터셋에서 정확한 예측의 비율을 계산합니다
- 분류 정확도를 계산하기 위해, 앞선 `argmax` 기반 예측 코드를 데이터셋의 모든 예제에 적용하고 다음과 같이 정확한 예측의 비율을 계산할 수 있습니다:

In [ ]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)

            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]  # 마지막 출력 토큰의 로짓
            predicted_labels = torch.argmax(logits, dim=-1)

            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    return correct_predictions / num_examples

- 다른 데이터셋들에 대한 분류 정확도를 계산하기 위해 함수를 적용해보겠습니다:

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 참고:
# 다음 줄들의 주석을 해제하면 Apple Silicon 칩에서 코드가 실행될 수 있으며,
# 이는 Apple CPU보다 약 2배 빠릅니다 (M3 MacBook Air에서 측정).
# 이 글을 쓰는 시점에서 PyTorch 2.4에서는 CPU와 MPS를 통해 얻은 결과가 동일했습니다.
# 하지만 PyTorch의 이전 버전에서는 MPS를 사용할 때 다른 결과를 관찰할 수 있습니다.

#if torch.cuda.is_available():
#    device = torch.device("cuda")
#elif torch.backends.mps.is_available():
#    device = torch.device("mps")
#else:
#    device = torch.device("cpu")
#print(f"Running on {device} device.")

model.to(device) # nn.Module 클래스들의 경우 model = model.to(device) 할당이 필요하지 않음

torch.manual_seed(123) # 훈련 데이터 로더의 셔플로 인한 재현성을 위해

train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"훈련 정확도: {train_accuracy*100:.2f}%")
print(f"검증 정확도: {val_accuracy*100:.2f}%")
print(f"테스트 정확도: {test_accuracy*100:.2f}%")

- 보시다시피, 아직 모델을 미세조정하지 않았기 때문에 예측 정확도가 별로 좋지 않습니다

- 미세조정(/훈련)을 시작하기 전에, 먼저 훈련 중에 최적화하고자 하는 손실 함수를 정의해야 합니다
- 목표는 모델의 스팸 분류 정확도를 최대화하는 것입니다; 하지만 분류 정확도는 미분 가능한 함수가 아닙니다
- 따라서 대신 분류 정확도를 최대화하기 위한 대리(proxy)로서 교차 엔트로피 손실을 최소화합니다 (이 주제에 대한 자세한 내용은 제가 무료로 제공하는 [딥러닝 입문](https://sebastianraschka.com/blog/2021/dl-course.html#l08-multinomial-logistic-regression--softmax-regression) 강의의 강의 8에서 배울 수 있습니다)

- `calc_loss_batch` 함수는 여기서 5장과 동일하지만, 모든 토큰 `model(input_batch)` 대신 마지막 토큰 `model(input_batch)[:, -1, :]`만 최적화하는 데 관심이 있다는 점이 다릅니다

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, -1, :]  # 마지막 출력 토큰의 로짓
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

`calc_loss_loader`는 5장과 정확히 동일합니다

In [ ]:
# 5장과 동일
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # num_batches가 데이터 로더의 배치 수를 초과하는 경우
        # 데이터 로더의 총 배치 수에 맞춰 배치 수를 줄임
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

- `calc_loss_loader`를 사용하여, 훈련을 시작하기 전의 초기 훈련, 검증, 테스트 세트 손실을 계산합니다

In [ ]:
with torch.no_grad(): # 아직 훈련하지 않으므로 효율성을 위해 그래디언트 추적 비활성화
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)

print(f"훈련 손실: {train_loss:.3f}")
print(f"검증 손실: {val_loss:.3f}")
print(f"테스트 손실: {test_loss:.3f}")

- 다음 섹션에서는 손실 값을 개선하고 결과적으로 분류 정확도를 향상시키기 위해 모델을 훈련합니다

## 6.7 지도 데이터로 모델 미세조정(Finetuning the model on supervised data)

- 이 섹션에서는 모델의 분류 정확도를 향상시키기 위한 훈련 함수를 정의하고 사용합니다
- 아래의 `train_classifier_simple` 함수는 5장에서 모델을 사전훈련하기 위해 사용한 `train_model_simple` 함수와 실질적으로 동일합니다
- 유일한 두 가지 차이점은 이제 다음과 같다는 것입니다:
  1. 보인 토큰 수 대신 보인 훈련 예제 수(`examples_seen`)를 추적합니다
  2. 각 에폭 후에 샘플 텍스트를 출력하는 대신 정확도를 계산합니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/training-loop.webp?1" width=500px>

In [ ]:
# 전체적으로 5장의 `train_model_simple`과 동일
def train_classifier_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                            eval_freq, eval_iter):
    # 손실과 보인 예제를 추적하기 위한 리스트 초기화
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1

    # 메인 훈련 루프
    for epoch in range(num_epochs):
        model.train()  # 모델을 훈련 모드로 설정

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # 이전 배치 반복에서의 손실 그래디언트 재설정
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # 손실 그래디언트 계산
            optimizer.step() # 손실 그래디언트를 사용하여 모델 가중치 업데이트
            examples_seen += input_batch.shape[0] # 새로움: 토큰 대신 예제 추적
            global_step += 1

            # 선택적 평가 단계
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"에폭 {epoch+1} (스텝 {global_step:06d}): "
                      f"훈련 손실 {train_loss:.3f}, 검증 손실 {val_loss:.3f}")

        # 각 에폭 후 정확도 계산
        train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)
        print(f"훈련 정확도: {train_accuracy*100:.2f}% | ", end="")
        print(f"검증 정확도: {val_accuracy*100:.2f}%")
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

- `train_classifier_simple`에서 사용되는 `evaluate_model` 함수는 5장에서 사용한 것과 동일합니다

In [ ]:
# 5장과 동일
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

- 훈련은 M3 MacBook Air 노트북 컴퓨터에서 약 5분, V100 또는 A100 GPU에서는 30초도 안 걸립니다

In [ ]:
import time

start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)

num_epochs = 5
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"훈련이 {execution_time_minutes:.2f}분 만에 완료되었습니다.")

- 5장과 유사하게, matplotlib을 사용하여 훈련 및 검증 세트의 손실 함수를 플롯합니다

In [ ]:
import matplotlib.pyplot as plt

def plot_values(epochs_seen, examples_seen, train_values, val_values, label="loss"):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # 에폭에 대한 훈련 및 검증 손실 플롯
    ax1.plot(epochs_seen, train_values, label=f"훈련 {label}")
    ax1.plot(epochs_seen, val_values, linestyle="-.", label=f"검증 {label}")
    ax1.set_xlabel("에폭")
    ax1.set_ylabel(label.capitalize())
    ax1.legend()

    # 보인 예제에 대한 두 번째 x축 생성
    ax2 = ax1.twiny()  # 동일한 y축을 공유하는 두 번째 x축 생성
    ax2.plot(examples_seen, train_values, alpha=0)  # 틱 정렬을 위한 비가시 플롯
    ax2.set_xlabel("보인 예제")

    fig.tight_layout()  # 공간 확보를 위해 레이아웃 조정
    plt.savefig(f"{label}-plot.pdf")
    plt.show()

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_losses))

plot_values(epochs_tensor, examples_seen_tensor, train_losses, val_losses)

- 위에서 하향 기울기를 기반으로, 모델이 잘 학습하고 있음을 볼 수 있습니다
- 또한 훈련 및 검증 손실이 매우 가깝다는 사실은 모델이 훈련 데이터를 과적합하는 경향이 없음을 나타냅니다
- 마찬가지로 아래에서 정확도를 플롯할 수 있습니다

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_accs))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_accs))

plot_values(epochs_tensor, examples_seen_tensor, train_accs, val_accs, label="accuracy")

- 위의 정확도 플롯을 기반으로, 모델이 에폭 4와 5 이후에 상대적으로 높은 훈련 및 검증 정확도를 달성하는 것을 볼 수 있습니다
- 하지만 앞서 훈련 함수에서 `eval_iter=5`를 지정했다는 점을 염두에 두어야 합니다. 이는 훈련 및 검증 세트 성능만 추정했음을 의미합니다
- 아래에서와 같이 전체 데이터셋에 대한 훈련, 검증, 테스트 세트 성능을 계산할 수 있습니다

In [ ]:
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)

print(f"훈련 정확도: {train_accuracy*100:.2f}%")
print(f"검증 정확도: {val_accuracy*100:.2f}%")
print(f"테스트 정확도: {test_accuracy*100:.2f}%")

- 훈련 및 검증 세트 성능이 실질적으로 동일함을 볼 수 있습니다
- 하지만 약간 낮은 테스트 세트 성능을 기반으로, 모델이 훈련 데이터와 학습률과 같은 하이퍼파라미터 조정에 사용된 검증 데이터를 매우 약간 과적합한다는 것을 볼 수 있습니다
- 이는 정상적인 현상이며, 이 격차는 모델의 드롭아웃 비율(`drop_rate`)이나 최적화기 설정의 `weight_decay`를 증가시킴으로써 잠재적으로 더 줄일 수 있습니다

## 6.8 LLM을 스팸 분류기로 사용하기(Using the LLM as a spam classifier)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/overview-4.webp" width=500px>

- 마지막으로, 미세조정된 GPT 모델을 실제로 사용해보겠습니다
- 아래의 `classify_review` 함수는 앞서 구현한 `SpamDataset`과 유사한 데이터 전처리 단계를 구현합니다
- 그런 다음, 함수는 모델에서 예측된 정수 클래스 라벨을 반환하고 해당 클래스 이름을 반환합니다

In [ ]:
def classify_review(text, model, tokenizer, device, max_length=None, pad_token_id=50256):
    model.eval()

    # 모델에 대한 입력 준비
    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[0]
    # 참고: 책에서 원래 실수로 pos_emb.weight.shape[1]로 작성되었습니다
    # 코드를 깨뜨리지는 않았지만 불필요한 잘림이 발생했을 것입니다 (1024 대신 768로)

    # 너무 긴 시퀀스는 잘라냄
    input_ids = input_ids[:min(max_length, supported_context_length)]
    assert max_length is not None, (
        "max_length가 지정되어야 합니다. 전체 모델 컨텍스트를 사용하려면, "
        "max_length=model.pos_emb.weight.shape[0]를 전달하세요."
    )
    assert max_length <= supported_context_length, (
        f"max_length ({max_length})가 모델의 지원되는 컨텍스트 길이 ({supported_context_length})를 초과합니다."
    )
    # 또는, 더 견고한 버전은 다음과 같으며, max_length=None 경우를 더 잘 처리합니다
    # max_len = min(max_length,supported_context_length) if max_length else supported_context_length
    # input_ids = input_ids[:max_len]
    
    # 가장 긴 시퀀스로 시퀀스를 패딩
    input_ids += [pad_token_id] * (max_length - len(input_ids))
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0) # 배치 차원 추가

    # 모델 추론
    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]  # 마지막 출력 토큰의 로짓
    predicted_label = torch.argmax(logits, dim=-1).item()

    # 분류 결과 반환
    return "spam" if predicted_label == 1 else "not spam"

- 아래에서 몇 가지 예제로 시도해보겠습니다

In [ ]:
text_1 = (
    "당신은 당첨자입니다. 특별히 선택되어"
    " $1000 현금 또는 $2000 상품을 받으실 수 있습니다."
)

print(classify_review(
    text_1, model, tokenizer, device, max_length=train_dataset.max_length
))

In [ ]:
text_2 = (
    "안녕, 오늘 저녁 약속 아직 유효한지"
    " 확인하려고 했어. 알려줘!"
)

print(classify_review(
    text_2, model, tokenizer, device, max_length=train_dataset.max_length
))

- 마지막으로, 다시 훈련하지 않고도 모델을 재사용하려는 경우를 대비해 모델을 저장해보겠습니다

In [ ]:
torch.save(model.state_dict(), "review_classifier.pth")

- 그런 다음, 새로운 세션에서 다음과 같이 모델을 로드할 수 있습니다

In [ ]:
model_state_dict = torch.load("review_classifier.pth", map_location=device, weights_only=True)
model.load_state_dict(model_state_dict)

## 요약 및 핵심 사항(Summary and takeaways)

- 분류 미세조정을 위한 자체 포함 스크립트인 [./gpt_class_finetune.py](./gpt_class_finetune.py) 스크립트를 참조하세요
- 연습 문제 해답은 [./exercise-solutions.ipynb](./exercise-solutions.ipynb)에서 찾을 수 있습니다
- 또한 관심 있는 독자들은 [부록 E](../../appendix-E)에서 저랭크 적응(LoRA)을 통한 매개변수 효율적 훈련에 대한 소개를 찾을 수 있습니다